# Notebook 2: DiffusionGemma in diffusers, with a drop-in top-k / certified scheduler

**Run on:** Colab **A100 40GB** (safest) or **L4** (4-bit may or may not fit). Budget: about 2 to 4 GPU-hours.

The sampler is exposed in Hugging Face diffusers: `DiffusionGemmaPipeline` runs the outer loop over 256-token canvases, and a swappable **scheduler** turns each step's logits into commits. `EntropyBoundScheduler` is the one matching the released checkpoint (entropy bound 0.1, temperature 0.8 to 0.4). Its `step()` takes raw logits `[batch, 256, vocab]` and returns the new canvas, the committed positions, the sampled tokens and probabilities, and `pred_logits` (temperature-scaled logits the model uses for self-conditioning on the next step).

**What this notebook does**

| # | Step | Gate |
|---|---|---|
| 1 | Load the model in 4-bit and run the stock pipeline | Does it fit and run? |
| 2 | Record per-step logits with a callback | Data for Notebooks 1 and 3 |
| 3 | Recover the scheduler's exact temperature schedule | Needed for faithful replays |
| 4 | Drop-in `TopKEntropyBoundScheduler`: modes `parent`, `full`, `topk`, `certified` | **Gate 1**: `full` and `certified` must reproduce the stock scheduler's commits exactly |
| 5 | Small GSM8K comparison across modes | First accuracy, step-count and speed numbers |
| 6 | Locate the two other places full-vocabulary work happens | Week-2 engineering targets |

**Where the full-vocabulary cost lives (important for the speed claim).** The scheduler is only one of three places:
1. **Scheduler** (sampling, entropy, commits): replaced in this notebook.
2. **Adaptive stopping** in the pipeline: it computes the mean entropy of `pred_logits` itself. To keep stopping unchanged, our scheduler returns the full temperature-scaled logits by default (`pred_logits_mode='full'`).
3. **Self-conditioning** inside the model: it receives `pred_logits` and does the probabilities x embedding-matrix product. Real speedups need a small patch there (section 6 finds it).

So this notebook first proves **decision equivalence**; the full speedup comes once (2) and (3) are patched.

## 1. Environment and install

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1
!free -g | head -2

In [ ]:
# DiffusionGemmaForBlockDiffusion needs a recent transformers (the diffusers-native checkpoint card asks for >= 5.12)
!pip install -q -U "transformers>=5.12" diffusers accelerate bitsandbytes datasets

In [ ]:
# from huggingface_hub import login; login()    # uncomment if the model page asks you to accept a license
import time, re, os, inspect, math, json
import torch, transformers, diffusers
from transformers import AutoConfig, AutoProcessor, BitsAndBytesConfig
print('torch', torch.__version__, '| transformers', transformers.__version__, '| diffusers', diffusers.__version__)

MODEL_ID = 'google/diffusiongemma-26B-A4B-it'
cfg = AutoConfig.from_pretrained(MODEL_ID)
tcfg = getattr(cfg, 'text_config', cfg)
print('architectures:', cfg.architectures)
def safe_get(c, key):
    '''Config value; per-layer (heterogeneous) attributes are reported per layer instead of raising.'''
    try:
        return getattr(c, key, None)
    except Exception:
        layers = getattr(c, 'per_layer_config', None)
        if layers is None:
            return '(per-layer attribute)'
        vals = [getattr(l, key, None) for l in layers]
        return f'per-layer, values {sorted(set(map(str, vals)))} across {len(vals)} layers'

for key in ['hidden_size', 'vocab_size', 'num_hidden_layers', 'num_key_value_heads', 'final_logit_softcapping']:
    print(f'{key:26s}', safe_get(tcfg, key))
for k, v in cfg.to_dict().items():
    if any(w in k.lower() for w in ['canvas', 'block', 'self_cond', 'expert']):
        print(f'{k:40s}', str(v)[:120])

Copy `hidden_size` and `vocab_size` into Notebook 1 (`D`, `V`).

## 2. Load the model (4-bit) and build the stock pipeline

In [ ]:
from transformers import DiffusionGemmaForBlockDiffusion
from diffusers import DiffusionGemmaPipeline, EntropyBoundScheduler

major, _ = torch.cuda.get_device_capability()
DTYPE = torch.bfloat16 if major >= 8 else torch.float16
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True)
t0 = time.time()
model = DiffusionGemmaForBlockDiffusion.from_pretrained(MODEL_ID, quantization_config=bnb, dtype=DTYPE,
                                                        device_map='auto', low_cpu_mem_usage=True)
processor = AutoProcessor.from_pretrained(MODEL_ID)
print(f'loaded in {time.time() - t0:.0f}s | GPU memory {torch.cuda.memory_allocated() / 2**30:.1f} GB')
offloaded = {k: v for k, v in (getattr(model, 'hf_device_map', None) or {}).items() if v in ('cpu', 'disk')}
if offloaded:
    print('WARNING: offloaded modules (timings meaningless):', list(offloaded)[:8])

N_STEPS = 48
stock = EntropyBoundScheduler(entropy_bound=0.1, t_max=0.8, t_min=0.4, num_inference_steps=N_STEPS)
pipe = DiffusionGemmaPipeline(model=model, scheduler=stock, processor=processor)

If loading runs out of memory: bitsandbytes may be leaving the MoE expert tensors unquantized. Move to the AWS L40S (48 GB) and load in bf16 or FP8 instead; the rest of the notebook is unchanged.

## 3. Stock run, with a callback that counts steps and records logits

The callback sees `canvas` and `logits` after every denoising step. Each call is one decoder forward pass, so the count gives tokens per forward directly.

In [ ]:
class StepRecorder:
    def __init__(self, keep_logits=0):
        self.keep = keep_logits
        self.reset()
    def reset(self):
        self.steps, self.timesteps, self.logits = 0, [], []
    def __call__(self, pipe, step, timestep, kw):
        self.steps += 1
        self.timesteps.append(int(timestep) if torch.is_tensor(timestep) else timestep)
        if len(self.logits) < self.keep and 'logits' in kw:
            self.logits.append(kw['logits'].detach().to('cpu', torch.float16))
        return {'canvas': kw['canvas']}

def generate(pipe, prompt, rec, gen_length=256, seed=0):
    rec.reset()
    torch.cuda.synchronize(); t0 = time.time()
    out = pipe(prompt=prompt, gen_length=gen_length, num_inference_steps=N_STEPS,
               generator=torch.Generator('cuda').manual_seed(seed),
               callback_on_step_end=rec, callback_on_step_end_tensor_inputs=['canvas', 'logits'])
    torch.cuda.synchronize()
    return out, time.time() - t0

rec = StepRecorder(keep_logits=12)
torch.cuda.reset_peak_memory_stats()
out, dt = generate(pipe, 'Why is the sky blue? Answer in one sentence.', rec)
n_tok = len(processor.tokenizer(out.texts[0])['input_ids'])
print(out.texts[0])
print(f'\n{dt:.2f}s | decoder steps {rec.steps} | ~{n_tok} text tokens | tokens per forward ~{n_tok / max(rec.steps, 1):.1f}')
print(f'peak GPU memory {torch.cuda.max_memory_allocated() / 2**30:.1f} GB')
print('logits shapes recorded:', [tuple(t.shape) for t in rec.logits[:3]])

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/dgemma_logits'
else:
    OUT_DIR = '/content/logits'
os.makedirs(OUT_DIR, exist_ok=True)
for i, L in enumerate(rec.logits):
    torch.save(L, f'{OUT_DIR}/step_{i:02d}.pt')
print(len(rec.logits), 'step tensors saved to', OUT_DIR)

## 4. Recover the stock scheduler's exact temperature per step

Rather than guess the schedule, run the stock `step()` on a test tensor at every timestep and read the temperature off `pred_logits` (it is `logits / temperature`). The printed list is the `TAU_TABLE` for Notebooks 1 and 3.

In [ ]:
def timesteps_of(sched, n=N_STEPS):
    try:
        sched.set_timesteps(n)
        ts = getattr(sched, 'timesteps', None)
        if ts is not None and len(ts) == n:
            return [int(t) for t in ts]
    except Exception as e:
        print('set_timesteps failed:', type(e).__name__, e)
    return list(range(n))

TIMESTEPS = timesteps_of(stock)
x = torch.randn(1, 8, 1024, device='cuda') * 3 + 1.0
sample = torch.randint(1024, (1, 8), device='cuda')
TAU_TABLE = []
for t in TIMESTEPS:
    o = stock.step(x, t, sample)
    TAU_TABLE.append(round((x / o.pred_logits.float()).median().item(), 6))
print('timesteps:', TIMESTEPS[:5], '...', TIMESTEPS[-3:])
print('TAU_TABLE =', TAU_TABLE)

Sanity check: the first value should be about 0.8 and the last about 0.4. If the ratio is not constant across the tensor, `pred_logits` is more than plain temperature scaling; print `inspect.getsource(EntropyBoundScheduler.step)` and adapt `TopKEntropyBoundScheduler._tau` below.

In [ ]:
print(inspect.getsource(EntropyBoundScheduler.step)[:3000])

## 5. The drop-in scheduler

`TopKEntropyBoundScheduler` subclasses the stock scheduler, so the pipeline treats it identically. Modes:

| mode | entropy for commits | sampling | purpose |
|---|---|---|---|
| `parent` | stock | stock | baseline |
| `full` | exact, full vocabulary | full vocabulary | re-implementation check: must match `parent` commits |
| `topk` | top-k with a chosen tail correction (`renorm`, `lo`, `mid`, `hi`) | top-k only | the approximate variants |
| `certified` | intervals + refinement (exact decisions) | exact tail-branch sampling | the paper's method |

`pred_logits_mode='full'` returns the full temperature-scaled logits, so self-conditioning and adaptive stopping are untouched and only the scheduler's own decisions are being tested.

In [ ]:
import torch.nn.functional as F
try:
    from diffusers.schedulers.scheduling_entropy_bound import EntropyBoundSchedulerOutput
except ImportError:
    from diffusers import EntropyBoundSchedulerOutput


def chunked_logsumexp(x, tau, chunk=32_768):
    out = None
    for s in range(0, x.shape[-1], chunk):
        part = torch.logsumexp(x[..., s:s + chunk].float() / tau, dim=-1)
        out = part if out is None else torch.logaddexp(out, part)
    return out

def exact_entropy(rows, tau):
    lp = torch.log_softmax(rows.float() / tau, dim=-1)
    return -(lp.exp() * lp).sum(-1)

def n_commit(e, b):
    s, _ = torch.sort(e)
    return int(((s.cumsum(0) - s) <= b).sum())

def commit_exact(e, b):
    order = torch.argsort(e, stable=True)
    keep = torch.zeros_like(e, dtype=torch.bool)
    keep[order[:n_commit(e, b)]] = True
    return keep

def topk_stats(logits, tau, k):
    Vv = logits.shape[-1]
    vals, idx = torch.topk(logits, k, dim=-1)
    lse = chunked_logsumexp(logits, tau)
    lp = vals.float() / tau - lse.unsqueeze(-1)
    p = lp.exp()
    h = -(p * lp).sum(-1)
    m = (1.0 - p.sum(-1)).clamp(0.0, 1.0)
    nmlm = -m * torch.log(m.clamp_min(1e-30))
    hi = h + nmlm + m * math.log(Vv - k)
    lo_simple = h + nmlm
    lo_tight = h - m * torch.log(torch.minimum(m, p[..., -1]).clamp_min(1e-30))
    return dict(idx=idx, p=p, m=m, lse=lse, h=h, hi=hi, lo=lo_tight, lo_simple=lo_simple)

def certified_commit_row(lo, hi, refine, b, chunk=8):
    lo, hi = lo.clone(), hi.clone()
    exact = torch.zeros_like(lo, dtype=torch.bool)
    refined = 0
    while True:
        if bool(exact.all()):
            return commit_exact(hi, b), refined
        n_lo, n_hi = n_commit(lo, b), n_commit(hi, b)
        if n_lo == n_hi:
            order = torch.argsort(hi, stable=True)
            S, R = order[:n_lo], order[n_lo:]
            if R.numel() == 0 or hi[S].max() < lo[R].min():
                keep = torch.zeros_like(exact); keep[S] = True
                return keep, refined
            amb = torch.cat([S[hi[S] >= lo[R].min()], R[lo[R] <= hi[S].max()]])
        else:
            amb = torch.argsort(lo, stable=True)[:n_lo + 1]
        amb = amb[~exact[amb]]
        if amb.numel() == 0:
            amb = (~exact).nonzero().squeeze(-1)
        if amb.numel() > chunk:
            amb = amb[torch.argsort(hi[amb] - lo[amb], descending=True)[:chunk]]
        e = refine(amb)
        lo[amb], hi[amb] = e, e
        exact[amb] = True
        refined += amb.numel()


class TopKEntropyBoundScheduler(EntropyBoundScheduler):
    def __init__(self, *args, mode='certified', k=64, tail='hi', pred_logits_mode='full',
                 tau_table=None, timesteps=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.mode, self.k, self.tail, self.pred_logits_mode = mode, k, tail, pred_logits_mode
        self.tau_by_t = dict(zip(timesteps, tau_table)) if tau_table and timesteps else None
        self.stats = []

    def _tau(self, timestep):
        t = int(timestep) if torch.is_tensor(timestep) else int(timestep)
        if self.tau_by_t and t in self.tau_by_t:
            return self.tau_by_t[t]
        n = self.config.num_inference_steps
        return self.config.t_max - (self.config.t_max - self.config.t_min) * min(t, n - 1) / max(n - 1, 1)

    def step(self, model_output, timestep, sample, entropy_bound=None, generator=None, return_dict=True):
        if self.mode == 'parent':
            return super().step(model_output, timestep, sample, entropy_bound=entropy_bound,
                                generator=generator, return_dict=return_dict)
        logits = model_output
        b = self.config.entropy_bound if entropy_bound is None else entropy_bound
        tau = self._tau(timestep)
        B, Cc, Vv = logits.shape
        g = generator if (generator is not None and generator.device == logits.device) else None
        refined = tail_rows = 0

        if self.mode == 'full':
            lp = torch.log_softmax(logits.float() / tau, -1)
            p = lp.exp()
            ent = -(p * lp).sum(-1)
            keep = torch.stack([commit_exact(ent[i], b) for i in range(B)])
            tok = torch.multinomial(p.view(-1, Vv), 1, generator=g).view(B, Cc)
            lse = torch.logsumexp(logits.float() / tau, -1)
        else:
            st = topk_stats(logits, tau, self.k)
            lse = st['lse']
            if self.mode == 'topk':
                if self.tail == 'renorm':
                    q = st['p'] / st['p'].sum(-1, keepdim=True)
                    ent = -(q * torch.log(q.clamp_min(1e-30))).sum(-1)
                else:
                    ent = {'lo': st['lo'], 'hi': st['hi'], 'mid': 0.5 * (st['lo'] + st['hi']), 'none': st['h']}[self.tail]
                keep = torch.stack([commit_exact(ent[i], b) for i in range(B)])
                pick = torch.multinomial(st['p'].reshape(-1, self.k), 1, generator=g).view(B, Cc)
                tok = st['idx'].gather(-1, pick.unsqueeze(-1)).squeeze(-1)
            elif self.mode == 'certified':
                rows = []
                for i in range(B):
                    refine = lambda pos, i=i: exact_entropy(logits[i, pos], tau)
                    kp, r = certified_commit_row(st['lo'][i], st['hi'][i], refine, b)
                    rows.append(kp); refined += r
                keep = torch.stack(rows)
                u = torch.rand(B, Cc, device=logits.device, generator=g)
                pick = torch.multinomial(st['p'].reshape(-1, self.k), 1, generator=g).view(B, Cc)
                tok = st['idx'].gather(-1, pick.unsqueeze(-1)).squeeze(-1)
                hit = (u < st['m']).nonzero()
                tail_rows = int(hit.shape[0])
                if tail_rows:
                    r0, r1 = hit[:, 0], hit[:, 1]
                    Lr = logits[r0, r1].float() / tau
                    Lr.scatter_(-1, st['idx'][r0, r1], float('-inf'))
                    tok[r0, r1] = torch.multinomial(torch.softmax(Lr, -1), 1, generator=g).squeeze(-1)
            else:
                raise ValueError(self.mode)

        noise = torch.randint(Vv, (B, Cc), device=logits.device, generator=g)
        prev = torch.where(keep, tok, noise)
        sampled_probs = torch.exp(logits.gather(-1, tok.unsqueeze(-1)).squeeze(-1).float() / tau - lse)
        if self.pred_logits_mode == 'full':
            pred_logits = logits.float() / tau
        else:  # 'topk_masked': self-conditioning and stopping see only the top-k (shows the naive failure mode)
            pred_logits = torch.full_like(logits, float('-inf'), dtype=torch.float32)
            vals, idx = torch.topk(logits, self.k, -1)
            pred_logits.scatter_(-1, idx, vals.float() / tau)
        self.stats.append(dict(t=int(timestep) if torch.is_tensor(timestep) else timestep,
                               committed=keep.sum(-1).float().mean().item(), refined=refined, tail_rows=tail_rows))
        out = EntropyBoundSchedulerOutput(prev_sample=prev, accepted_index=keep, sampled_tokens=tok,
                                          sampled_probs=sampled_probs, pred_logits=pred_logits)
        return out if return_dict else out.to_tuple()

def make_sched(mode, **kw):
    return TopKEntropyBoundScheduler(entropy_bound=0.1, t_max=0.8, t_min=0.4, num_inference_steps=N_STEPS,
                                     mode=mode, tau_table=TAU_TABLE, timesteps=TIMESTEPS, **kw)

### Gate 1: do `full` and `certified` reproduce the stock commits exactly?

Commit sets depend only on entropies, not on random draws, so on the same recorded logits the `accepted_index` must match the stock scheduler position for position. `topk` modes are expected to differ; that difference is the effect being studied.

In [ ]:
def commits_of(sched, L, t):
    return sched.step(L, t, torch.zeros(L.shape[:2], dtype=torch.long, device=L.device)).accepted_index

rows = []
for i, L in enumerate(rec.logits):
    if L.dim() != 3 or L.shape[1] < 2:
        continue
    L = L.to('cuda')
    t = TIMESTEPS[min(i, len(TIMESTEPS) - 1)]
    ref = commits_of(stock, L, t)
    for name, sch in [('full', make_sched('full')), ('certified k64', make_sched('certified', k=64)),
                      ('certified k256', make_sched('certified', k=256)),
                      ('topk64 renorm', make_sched('topk', k=64, tail='renorm')),
                      ('topk64 hi', make_sched('topk', k=64, tail='hi'))]:
        acc = commits_of(sch, L, t)
        rows.append(dict(step=i, variant=name, identical=bool((acc == ref).all()),
                         n_ref=int(ref.sum()), n_var=int(acc.sum()),
                         refined=sch.stats[-1]['refined'] if sch.stats else 0))
    del L

import pandas as pd
gate = pd.DataFrame(rows)
display(gate.groupby('variant')[['identical', 'n_ref', 'n_var', 'refined']].mean().round(3))
print('GATE 1 PASSED' if gate[gate.variant.str.startswith(('full', 'certified'))].identical.all()
      else 'GATE 1 FAILED: inspect the stock step source above (tie-breaking, temperature, or entropy definition)')

## 6. Small GSM8K comparison across modes

20 problems, greedy-style evaluation by the last number in the answer. This is a smoke test of accuracy and step counts, not the final evaluation (that uses 250 problems and 3 seeds).

In [ ]:
from datasets import load_dataset
gsm = load_dataset('openai/gsm8k', 'main', split='test[:20]')

def last_number(s):
    nums = re.findall(r'-?\d[\d,]*\.?\d*', s)
    return nums[-1].replace(',', '').rstrip('.') if nums else None

def gold(ans):
    return ans.split('####')[-1].strip().replace(',', '')

variants = {
    'stock':             stock,
    'certified k64':     make_sched('certified', k=64),
    'topk64 renorm':     make_sched('topk', k=64, tail='renorm'),
}
results = []
for name, sch in variants.items():
    pipe.scheduler = sch
    for j, ex in enumerate(gsm):
        prompt = ex['question'] + '\nSolve step by step, then give the final answer as a number.'
        out, dt = generate(pipe, prompt, rec, gen_length=512, seed=j)
        pred = last_number(out.texts[0])
        results.append(dict(variant=name, q=j, correct=pred == gold(ex['answer']), steps=rec.steps, seconds=dt))
pipe.scheduler = stock
res = pd.DataFrame(results)
display(res.groupby('variant')[['correct', 'steps', 'seconds']].mean().round(3))

Expect `certified` to match `stock` in accuracy and step count up to sampling noise (its decisions are identical and its samples come from the same distribution); `topk64 renorm` may finish in fewer steps with lower accuracy. With only 20 problems, differences of a few points are noise.

## 7. Find the other two full-vocabulary costs (week-2 targets)

Prints where the pipeline computes the stopping entropy and where the model does self-conditioning, so you know exactly what to patch for end-to-end speed.

In [ ]:
import importlib
pdg = importlib.import_module(type(pipe).__module__)
print('pipeline file:', inspect.getsourcefile(type(pipe)))
for i, line in enumerate(inspect.getsource(pdg).splitlines(), 1):
    if re.search(r'confidence_threshold|entropy|softmax|self_cond', line):
        print(f'{i:5d}: {line.strip()[:120]}')

mdl = importlib.import_module(type(model).__module__)
print('\nmodel file:', inspect.getsourcefile(type(model)))
for i, line in enumerate(inspect.getsource(mdl).splitlines(), 1):
    if re.search(r'self_cond|softmax|embed_tokens\.weight|@ *self\.|matmul', line):
        print(f'{i:5d}: {line.strip()[:120]}')

## 8. Summary for the week-2 plan

| Check | Result |
|---|---|
| Loads without offload on this GPU? Memory after load / peak | |
| Stock output sensible? Steps per canvas? | |
| TAU_TABLE endpoints about 0.8 and 0.4? | |
| Gate 1: `full` and `certified` commits identical to stock? | |
| Mean refined slots per step (certified k64 / k256) | |
| GSM8K-20: stock vs certified vs topk-renorm (accuracy, steps) | |
| Stopping-entropy and self-conditioning code located? | |

## 9. Optional: vLLM throughput sweep (AWS L40S, fresh runtime)

For the batch-size crossover (H3). FP8 weights need about 26 GB, so use a 48 GB L40S, not a Colab L4.

In [ ]:
RUN_VLLM = False
if RUN_VLLM:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'vllm'], check=True)
    from vllm import LLM, SamplingParams
    llm = LLM(model=MODEL_ID, quantization='fp8', max_model_len=4096, gpu_memory_utilization=0.90)
    sp = SamplingParams(max_tokens=512)
    for batch in [1, 2, 4, 8, 16, 32, 64]:
        t0 = time.time(); outs = llm.generate(['Explain how a rainbow forms, in detail.'] * batch, sp); dt = time.time() - t0
        n = sum(len(o.outputs[0].token_ids) for o in outs)
        print(f'batch {batch:3d}: {n / dt:8.0f} tok/s total, {n / dt / batch:7.0f} tok/s per request')